## Importar bibliotecas

En esta etapa construiremos complejos simpliciales y obtendremos los pares de nacimiento-muerte de las las clases de homología persistente a partir de datos espaciales.

Utilizaremos las siguientes bibliotecas:

- `os` para manejo de archivos y rutas.
- `numpy` para operaciones numéricas.
- `pandas` para manipulación de datos tabulares.
- `matplotlib` para visualización de resultados.
- `gudhi` para la construcción de complejos de Vietoris–Rips y homología persistente.


In [ ]:
# pip install gudhi

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gudhi as gd

Primero trabajaremos con una única muestra para entender cada paso del proceso. Una vez comprendido el pipeline completo, lo automatizaremos para procesar múltiples muestras de manera simultánea.

## Cargar datos

El primer paso consiste en cargar las coordenadas espaciales de las células que fueron obtenidas durante el proceso de segmentación.

Trabajaremos con un archivo CSV que contiene, entre otras variables, las coordenadas de los centroides celulares:

- `X_centroid`
- `Y_centroid`

Cada fila representa una célula y sus coordenadas dentro de la imagen.

A partir de estas coordenadas construiremos una **nube de puntos** (*point cloud*), que será la entrada para el análisis topológico.

La siguiente celda carga el archivo, extrae las coordenadas y las organiza en una matriz denominada `puntos`, donde cada fila corresponde a una célula y cada columna a una coordenada espacial.


In [ ]:
ruta = "https://raw.githubusercontent.com/HaydeePeruyero/tda-cell-patterns-workshop/main/data/high_grade_displasia_FAAGSCC_13_HGD_15.csv"

df = pd.read_csv()

x = df[""].to_numpy()
y = df[""].to_numpy()

puntos = np.column_stack((, ))

print(f"Número de puntos: {len()}")

## Visualización de la nube de puntos

Una vez cargadas las coordenadas, es conveniente visualizar la distribución espacial de las células antes de realizar cualquier análisis topológico.

Cada punto de la gráfica representa una célula identificada en el tejido. Esta representación constituye la nube de puntos (*point cloud*) sobre la cual construiremos posteriormente el complejo de Vietoris–Rips.

La siguiente celda genera una visualización simple de las coordenadas celulares.

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(, , s=5)

plt.title("Point Cloud (centroides celulares)")
plt.xlabel("X")
plt.ylabel("Y")

plt.show()

## Construcción del complejo de Vietoris–Rips

A partir de la nube de puntos se construye un complejo de Vietoris–Rips para una secuencia de valores del parámetro de escala $\varepsilon$. Al variar este parámetro, la conectividad entre las células cambia progresivamente, generando una familia de complejos simpliciales que describe la organización espacial de la muestra a múltiples escalas.

Esta secuencia de complejos constituye una filtración, sobre la cual se calcula posteriormente la homología persistente.

En este ejemplo utilizaremos inicialmente:

- $\varepsilon = 100$  (solo para este ejemplo)
- dimensión máxima = 2

La biblioteca GUDHI permite construir el complejo y almacenarlo en una estructura denominada `simplex_tree`, que posteriormente utilizaremos para calcular homología persistente.

## Advertencia

Para este taller **no utilices un valor de $\varepsilon$ mayor a 100** al construir el complejo de Vietoris–Rips.

El número de símplices crece rápidamente conforme aumenta \($ \varepsilon$ ), lo que incrementa de manera considerable el consumo de memoria y tiempo de cómputo. En el servidor utilizado para este curso, ejecutar el análisis con valores mayores a 100 podría afectar su rendimiento o incluso provocar que el proceso falle.

En otros proyectos o en servidores con mayores recursos computacionales, es posible trabajar con valores de \(\varepsilon\) superiores si el análisis lo requiere. Sin embargo, **para este taller se solicita limitar el valor máximo de \(\varepsilon\) a 100** con el fin de preservar la estabilidad y disponibilidad del servidor para todos los usuarios.

In [ ]:
radio = 100

rips_complex = gd._______(
    points=______,
    max_edge_length=_______
)
#NO CAMBIAR LA DIMENSIÓN MÁXIMA
simplex_tree = rips_complex.create_simplex_tree(
    max_dimension=2
)

print("Número de simplices:", simplex_tree._ces())

## Visualización del complejo de Vietoris–Rips

Aunque el complejo de Vietoris–Rips se define como un objeto combinatorio, resulta útil visualizar sus conexiones para comprender cómo la información geométrica de la nube de puntos se transforma en una estructura topológica.

Esta representación permite identificar visualmente regiones altamente conectadas, posibles agrupamientos y estructuras que podrían dar origen a ciclos topológicos.

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(, , color="black", s=5)

for simplex in simplex_tree.g______(1):
    if len(simplex[___]) == 2:
        i, j = simplex[0]

        plt.plot(
            [x[i], x[j]],
            [y[i], y[j]],
            color="gray",
            linewidth=0.5
        )

plt.title(f"Complejo de Rips (r={radio})")
plt.xlabel("X")
plt.ylabel("Y")

plt.show()

## Homología persistente

Hasta ahora hemos construido y visualizado un complejo de Vietoris–Rips para un valor fijo del parámetro de escala $r$.

Sin embargo, una de las ideas centrales de TDA es que la elección de una única escala puede ocultar información relevante. En lugar de analizar un solo complejo, la homología persistente estudia cómo evolucionan las estructuras topológicas cuando el parámetro de escala aumenta progresivamente, es decir, estudia un complejo simplicial filtrado.

La homología persistente registra estos eventos mediante pares $(\text{birth}, \text{death})$, que indican la escala en la que una clase de homología aparece y la escala en la que desaparece.

En este taller nos centraremos principalmente en las dimensiones 0 y 1 de la homología persistente:

- $H_0$: pares nacimiento–muerte correspondientes a componentes conexas.
- $H_1$: pares nacimiento–muerte correspondientes a ciclos o agujeros unidimensionales.

La biblioteca GUDHI permite calcular automáticamente estos pares o características a partir del complejo construido previamente.

In [ ]:
diag = simplex_tree.____()

print("Primeros valores:")
print(diag[:10])

### Generación del diagrama

El siguiente código permite visualizar el diagrama de persistencia obtenido a partir del complejo de Vietoris–Rips construido previamente.


In [ ]:
plt.figure(figsize=(6,6))
gd._______persistence_diagram(_______)
plt.title("Diagrama de persistencia")
plt.show()

## Guardar resultados

El diagrama de persistencia es una representación gráfica muy útil para visualizar las características topológicas de una muestra. Sin embargo, para realizar comparaciones sistemáticas entre múltiples muestras resulta conveniente almacenar esta información en formato tabular.

Por ello, extraeremos para cada característica topológica:

- su dimensión, que indica el tipo de característica topológica que representa (0 para componentes conexas, 1 para ciclos)
- el valor de nacimiento (*birth*),
- el valor de muerte (*death*).

Esta tabla será la entrada para las siguientes etapas del análisis, donde calcularemos distancias entre diagramas de persistencia y construiremos matrices de similitud entre muestras.


In [ ]:
output = pd.DataFrame(
    [[dim, b, d] for dim, (b, d) in diag if dim <= 2],
    columns=["dimension", "birth", "death"]
)

output.head()

## Resumen

A partir de las coordenadas celulares hemos construido:

1.  Una nube de puntos.
2.  Un complejo de Vietoris–Rips.
3.  Pares nacimiento–muerte de las clases de homología persistente..
4.  Un diagrama de persistencia.
5.  Una tabla con las características topológicas detectadas.

Hasta este punto hemos trabajado únicamente con una muestra. Sin embargo, en aplicaciones reales normalmente disponemos de múltiples muestras biológicas y necesitamos repetir exactamente el mismo procedimiento para cada una de ellas.

Afortunadamente, no es necesario ejecutar manualmente el análisis archivo por archivo. Podemos automatizar el proceso utilizando ciclos (`for`) y aplicar el pipeline completo a todos los archivos de una carpeta.

En teoría, todo lo necesario para construir este análisis ya ha sido presentado en las secciones anteriores. El siguiente ejercicio consiste en completar un script que procese automáticamente múltiples muestras y genere un diagrama de persistencia para cada una de ellas.

##  Ejercicio: procesar múltiples muestras

El ejercicio consiste en automatizar este proceso para todas las muestras disponibles. Para cada archivo `.csv`, deberán construir la nube de puntos, generar el complejo de Vietoris–Rips, obtener los pares nacimiento–muerte del diagrama de persistencia y guardar los resultados en una carpeta de salida.


Complete los espacios faltantes en el código para procesar las $n$ muestras y guardar un archivo de resultados por cada una.

**Nota:** El código mostrado es solo una guía para organizar el proceso. Puede implementar la solución de otra manera, siempre y cuando procese todos los archivos de la carpeta indicada y guarde los resultados correspondientes para cada muestra.


In [ ]:
import os
import requests
import numpy as np
import pandas as pd
import gudhi as gd

# ============================================================
# 1. OBTENER LISTA DE ARCHIVOS DESDE GITHUB
# ============================================================

api_url = (
    "https://api.github.com/repos/"
    "HaydeePeruyero/tda-cell-patterns-workshop/contents/data"
)

contenido = requests.get(api_url).json()

archivos = sorted([
    f["name"]
    for f in contenido
    if f["name"].endswith(".csv")
])

print("Archivos encontrados:")
for archivo in archivos:
    print("-", archivo)

# ============================================================
# 2. RUTA RAW DE GITHUB
# ============================================================

ruta = (
    "https://raw.githubusercontent.com/"
    "HaydeePeruyero/tda-cell-patterns-workshop/main/data/"
)

# ============================================================
# 3. CREAR CARPETA DE SALIDA
# ============================================================

os.makedirs("resultados", exist_ok=True)

# ============================================================
# 4. PROCESAR TODOS LOS ARCHIVOS
# ============================================================

for archivo in archivos:

    print(f"\nProcesando: {archivo}")

    # -------------------------
    # Cargar datos
    # -------------------------

    df = __________________

    # -------------------------
    # Coordenadas
    # -------------------------

    x = __________________
    y = __________________

    # -------------------------
    # Nube de puntos
    # -------------------------

    puntos = __________________

    # -------------------------
    # Complejo de Rips
    # -------------------------

    rips_complex = _____________

    simplex_tree = ______________

    # -------------------------
    # Homología persistente
    # -------------------------

    diag = __________________

    # -------------------------
    # Convertir a DataFrame
    # -------------------------

    output = pd.DataFrame(
        [[dim, b, d] for dim, (b, d) in diag if dim <= 2],
        columns=["dimension", "birth", "death"]
    )

    # -------------------------
    # Guardar resultado
    # -------------------------

    nombre_salida = __________________

    output.to_csv(
        os.path.join("resultados", nombre_salida),
        index=False
    )

    print(f"Guardado: {nombre_salida}")

# Distancia de Wasserstein



## ¿Por qué necesitamos una distancia?

Hasta ahora hemos transformado las coordenadas celulares en diagramas de persistencia mediante complejos de Vietoris–Rips y homología persistente.

Cada diagrama de persistencia resume la estructura topológica de una muestra biológica. Sin embargo, para responder preguntas biológicas necesitamos una forma de comparar dichos diagramas de manera cuantitativa.

Por ejemplo:

- ¿Dos muestras tumorales presentan organizaciones espaciales similares?
- ¿Las muestras de pacientes con anemia de Fanconi son más parecidas entre sí que a las muestras no FA?
- ¿Existen diferencias topológicas entre epitelio y estroma?

Para responder estas preguntas necesitamos definir una métrica entre diagramas de persistencia.

## Comparando muestras biológicas

Supongamos que hemos calculado diagramas de persistencia para distintas muestras:

- Carcinoma
- Displasia
- Estroma adyacente

Cada muestra produce un diagrama que resume su organización espacial celular.

El siguiente paso consiste en comparar estos diagramas utilizando la distancia de Wasserstein.

``` text
Carcinoma  ─────┐
                ├── Wasserstein ──► Distancia(Carcinoma, Displasia)
Displasia  ─────┘


Carcinoma  ─────┐
                ├── Wasserstein ──► Distancia(Carcinoma, Estroma)
Estroma    ─────┘
```

Si obtenemos, por ejemplo,

``` text
Distancia(Carcinoma, Displasia) = 15

Distancia(Carcinoma, Estroma) = 48
```

podemos interpretar que el patrón topológico observado en la muestra de carcinoma es más parecido al de la displasia que al del estroma adyacente.

Repitiendo este procedimiento para todos los pares de muestras obtenemos una matriz de distancias, la cual servirá posteriormente como entrada para el análisis de clustering jerárquico.


## Importar bibliotecas

Utilizaremos las siguientes bibliotecas durante esta sección del análisis:

- `os` para la gestión de archivos y directorios.
- `numpy` para operaciones numéricas y manejo de arreglos.
- `pandas` para la manipulación de datos en formato tabular.
- `gudhi.wasserstein` para el cálculo de distancias entre diagramas de persistencia.


In [ ]:
# pip install POT

In [ ]:
import os
import numpy as np
import pandas as pd
import gudhi.wasserstein as gw
import ot

## Definir la carpeta de trabajo

Los diagramas de persistencia generados en la sección anterior fueron almacenados como archivos CSV.

El siguiente paso consiste en indicar la carpeta donde se encuentran dichos archivos para poder cargarlos y compararlos entre sí mediante la distancia de Wasserstein.

In [ ]:
ruta = "" # cambiar dependiendo del usuario

## Obtener la lista de diagramas

Una vez definida la carpeta de trabajo, podemos identificar automáticamente todos los diagramas de persistencia disponibles.

La siguiente celda busca todos los archivos con extensión `.csv`, los ordena alfabéticamente y muestra la lista de muestras que serán utilizadas en el análisis.

Esto resulta especialmente útil cuando se trabaja con decenas o cientos de muestras, ya que evita tener que especificar manualmente cada archivo.

In [ ]:
archivos = sorted(
    [f for f in os.listdir(ruta) if f.endswith(".csv")]
)

print("Archivos encontrados:")

for i, archivo in enumerate(archivos):
    print(f"{i}: {archivo}")

Archivos encontrados:
0: high_grade_displasia_FAAGSCC_13_HGD_15_diag.csv
1: high_grade_displasia_FAAGSCC_13_HGD_2_diag.csv
2: high_grade_displasia_FAAGSCC_13_HGD_7_diag.csv
3: invasive_carcinoma_AGSCC_1_IC_18_diag.csv
4: invasive_carcinoma_AGSCC_1_IC_24_diag.csv
5: invasive_carcinoma_AGSCC_1_IC_28_diag.csv
6: light_grade_displasia_F82P1_LGD_5_diag.csv
7: light_grade_displasia_F82P1_LGD_6_diag.csv
8: stroma_ad_high_grade_displasia_FAAGSCC_13_HGD_4_diag.csv
9: stroma_ad_high_grade_displasia_FAAGSCC_13_HGD_7_diag.csv
10: stroma_ad_invasive_carcinoma_AGSCC_1_IC_111_diag.csv
11: stroma_ad_invasive_carcinoma_AGSCC_1_IC_99_diag.csv
12: stroma_ad_light_grade_displasia_F33P1_LGD_7_diag.csv
13: stroma_ad_light_grade_displasia_F81P1_LGD_1_diag.csv


## Carga de los diagramas de persistencia

Una vez obtenidos los diagramas de persistencia de cada muestra, el siguiente paso consiste en cargarlos en memoria para poder compararlos mediante la distancia de Wasserstein.

Cada archivo CSV contiene las características topológicas identificadas durante el cálculo de homología persistente. A partir de estos archivos construiremos dos listas independientes:

- `diag_dim0`: diagramas correspondientes a los pares nacimiento–muerte asociados con las clases de homología de $H_0$ (componentes conexas).
- `diag_dim1`: diagramas correspondientes a pares nacimiento–muerte asociados a clases de homología de $H_1$ (ciclos).

En el caso de $H_1$, eliminaremos las características con muerte infinita. Esto se hace para simplificar el cálculo de las distancias y evitar problemas asociados al manejo de valores infinitos.

Recordemos que, al construir una filtración de Vietoris–Rips con un radio suficientemente grande, todas las componentes conexas terminan fusionándose en una sola. Como consecuencia, en dimensión $0$ suele existir una única componente que persiste hasta infinito, representando la componente conexa final del espacio. Dado que esta característica aparece de manera sistemática en todas las muestras, su presencia no aporta información discriminante para la comparación entre diagramas.

Por esta razón, es común ignorar los puntos con muerte infinita al calcular distancias entre diagramas de persistencia.


In [ ]:
diag_dim0 = []
diag_dim1 = []

for archivo in __________:

    df = pd.read_csv(os.path.join(ruta, archivo))

    d0 = (
        df[
            (df["__________"] == __)
            & (____________________)
        ][["__________", "__________"]]
        .to_numpy()
    )

    d1 = (
        df[
            (df["__________"] == __)
            & (____________________)
        ][["__________", "__________"]]
        .to_numpy()
    )

    diag_dim0.append(____)
    diag_dim1.append(____)

## Inicialización de las matrices de distancia

Una vez cargados todos los diagramas de persistencia, necesitamos una estructura donde almacenar las distancias de Wasserstein entre cada par de muestras.

Si disponemos de $n$ muestras, construiremos dos matrices cuadradas de tamaño $n \times n$:

- `dist_dim0`: almacenará las distancias entre diagramas de dimensión $0$.
- `dist_dim1`: almacenará las distancias entre diagramas de dimensión $1$.

La entrada $(i,j)$ de cada matriz contendrá la distancia entre las muestras $i$ y $j$.

Por construcción, estas matrices serán simétricas, ya que la distancia de Wasserstein cumple

$$W(D_i,D_j)=W(D_j,D_i).$$

Además, la diagonal principal contendrá únicamente ceros, pues la distancia entre un diagrama y sí mismo es igual a cero.


In [ ]:
n = len(archivos)

dist_dim0 = np.zeros((n, n))
dist_dim1 = np.zeros((n, n))

## Cálculo de las distancias de Wasserstein

Una vez cargados todos los diagramas de persistencia, podemos comparar cada muestra con todas las demás mediante la distancia de Wasserstein.

El procedimiento consiste en recorrer todos los pares de diagramas y calcular la distancia correspondiente. El resultado se almacena en las matrices `dist_dim0` y `dist_dim1`, que contendrán las distancias para $dim 0$ y $dim 1$, respectivamente.

En este trabajo utilizaremos la distancia de Wasserstein de orden 1. Sin embargo, existen otras métricas ampliamente utilizadas para comparar diagramas de persistencia, como la distancia de cuello de botella (*bottleneck distance*), la cual también está implementada en GUDHI.

In [ ]:
for i in range(n):

    for j in range(n):

        dist_dim0[i, j] =  gw.wasserstein_distance(
            ____________,
            ____________,
            order=__
        )

        dist_dim1[i, j] =  gw.wasserstein_distance(
            ____________,
            ____________,
            order=__
        )

        # Alternativa:
        # dist_dim0[i, j] = gd.bottleneck_distance(
        #     diag_dim0[i],
        #     diag_dim0[j]
        # )

## Construcción de las matrices de distancia

Hasta este punto hemos calculado las distancias de Wasserstein entre todos los pares de muestras y las hemos almacenado en arreglos de NumPy.

Para facilitar su exploración e interpretación, convertiremos estas matrices a objetos `DataFrame` de pandas. Esto permitirá etiquetar filas y columnas con los nombres de los archivos correspondientes, haciendo más sencilla la identificación de las comparaciones entre muestras.

Cada entrada de la matriz representa la distancia topológica entre dos muestras:

- Valores cercanos a cero indican diagramas muy similares.
- Valores grandes indican diferencias más pronunciadas en la organización topológica.

La diagonal principal debe contener únicamente ceros, ya que cada muestra se compara consigo misma.


In [ ]:
df_dist0 = pd.DataFrame(
    dist_dim0,
    index=________,
    columns=_______
)

df_dist1 = pd.DataFrame(
    dist_dim1,
    index=,
    columns=
)

print(df_dist0)
print(df_dist1)

                                                    high_grade_displasia_FAAGSCC_13_HGD_15_diag.csv  \
high_grade_displasia_FAAGSCC_13_HGD_15_diag.csv                                            0.000000   
high_grade_displasia_FAAGSCC_13_HGD_2_diag.csv                                          7710.785536   
high_grade_displasia_FAAGSCC_13_HGD_7_diag.csv                                          6388.711787   
invasive_carcinoma_AGSCC_1_IC_18_diag.csv                                               4712.810788   
invasive_carcinoma_AGSCC_1_IC_24_diag.csv                                               2600.703114   
invasive_carcinoma_AGSCC_1_IC_28_diag.csv                                               3324.071489   
light_grade_displasia_F82P1_LGD_5_diag.csv                                              7262.814359   
light_grade_displasia_F82P1_LGD_6_diag.csv                                              5994.057292   
stroma_ad_high_grade_displasia_FAAGSCC_13_HGD_4...                       

## Guardado de resultados

En este punto ya hemos construido las matrices de distancias de Wasserstein entre todas las muestras analizadas.

El siguiente paso consiste en almacenar estos resultados, ya que serán utilizados posteriormente como entrada para el análisis de clustering jerárquico.

De esta forma, se separa el proceso de cómputo (cálculo de diagramas y distancias) del proceso de análisis exploratorio, lo que permite reutilizar las matrices sin necesidad de recalcularlas.

In [ ]:
# Carpeta de salida
salida = os.path.join(ruta, "distancias")
os.makedirs(salida, exist_ok=True)

In [ ]:
df_dist0.to_csv(
    os.path.join(ruta, "wasserstein_dim0.csv")
)

df_dist1.to_csv(
    os.path.join(ruta, "wasserstein_dim1.csv")
)

print("Matrices guardadas.")

Matrices guardadas.


Al finalizar, tendremos una matriz de distancias de Wasserstein que cuantifica la similitud topológica entre todas las muestras analizadas. En la siguiente sección utilizaremos estas matrices para construir dendrogramas y clustermaps mediante clustering jerárquico.

# Clustering

Una vez construidas las matrices de distancia de Wasserstein, el siguiente paso consiste en analizar la estructura global de similitud entre muestras.

El objetivo ahora es pasar de comparaciones pares (muestra vs muestra) a una representación global que permita identificar grupos de comportamiento topológico similar.

Este tipo de análisis es fundamental porque nos permite responder preguntas como:

- ¿Qué muestras comparten patrones espaciales similares?
- ¿Existen agrupamientos asociados a condiciones biológicas (FA vs no-FA, carcinoma vs estroma, etc.)?
- ¿La estructura del microambiente tumoral induce separaciones naturales en el espacio de muestras?

## Métodos de clustering

Existen múltiples enfoques para realizar agrupamiento a partir de una matriz de distancias, entre los más comunes:

- K-means (requiere espacio euclidiano explícito)
- DBSCAN (basado en densidad)
- Clustering espectral
- Clustering jerárquico

En este trabajo utilizaremos **clustering jerárquico aglomerativo**, ya que:

- trabaja directamente con matrices de distancia,
- no requiere asumir una forma predefinida de los clusters,
- y permite visualizar la estructura de agrupamiento en diferentes escalas.

## Clustering jerárquico

El clustering jerárquico construye una estructura en forma de árbol (dendrograma) donde:

- cada muestra inicia como un cluster individual,
- los clusters más similares se van fusionando iterativamente,
- hasta formar una única estructura global.

Este proceso puede visualizarse como una "jerarquía de similitud" entre muestras.

### Método de enlace (linkage)

Dentro del clustering jerárquico existen distintas formas de definir la distancia entre clusters:

- single linkage (mínimo)
- complete linkage (máximo)
- average linkage (promedio)

En este trabajo utilizaremos **average linkage**, ya que ha mostrado un comportamiento más estable y consistente en nuestros experimentos, al suavizar la influencia de valores extremos en las distancias.


## Resumen 

A partir de estas matrices construiremos dendrogramas y clustermaps que nos permitirán visualizar la estructura global de similitud entre muestras y explorar posibles asociaciones con variables biológicas como:

- Grado histopatológico (LGD, HGD, IC).
- Estado de anemia de Fanconi (FA y no-FA).
- Compartimento tisular (epitelio y estroma).


## Importar bibliotecas

Comenzaremos importando las bibliotecas necesarias para realizar el análisis de clustering y la visualización de los resultados.

Utilizaremos:

- `pandas` para manipular matrices de distancia.
- `seaborn` para generar clustermaps.
- `matplotlib` para personalizar las figuras.
- `os` para gestionar archivos y directorios.
- `re` para identificar automáticamente características de las muestras a partir de sus nombres.

In [ ]:
import os
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

## Definir directorios de trabajo

Las matrices de distancia calculadas en la sección anterior fueron almacenadas como archivos `.csv`.

El siguiente paso consiste en definir la carpeta donde se encuentran dichas matrices y crear un directorio donde se guardarán automáticamente las visualizaciones generadas durante el análisis.


In [ ]:
# Ruta de las matrices de distancia
ruta = ""

# Carpeta de salida
salida = os.path.join(ruta, "visualizacion")
os.makedirs(salida, exist_ok=True)

## Clasificación automática de las muestras

Para facilitar la interpretación biológica de los agrupamientos, utilizaremos funciones auxiliares que permiten identificar automáticamente diferentes características de cada muestra a partir de su nombre.

En particular, clasificaremos cada muestra según:

- El tipo de lesión histopatológica.
- El estado de anemia de Fanconi.
- El compartimento tisular al que pertenece.

Estas categorías se utilizarán posteriormente para colorear las filas y columnas del clustermap.

In [ ]:
def roi(nombre):

    nombre = nombre.lower()

    if "invasive_carcinoma" in nombre:
        return "IC"

    elif "high_grade_displasia" in nombre:
        return "HGD"

    elif "light_grade_displasia" in nombre:
        return "LGD"

    return "Other"

In [ ]:
def fanconi(nombre):

    nombre = nombre.upper()

    if re.search(r"_FA", nombre):
        return "Fanconi"

    if re.search(r"_F\d", nombre):
        return "Fanconi"

    return "No Fanconi"

In [ ]:
def fanconi(nombre):

    nombre = nombre.upper()

    if re.search(r"_FA", nombre):
        return "Fanconi"

    if re.search(r"_F\d", nombre):
        return "Fanconi"

    return "No Fanconi"

In [ ]:
def tejido(nombre):

    if nombre.startswith("stroma_ad_"):
        return "Stroma"

    return "Epithelium"

## Definición de colores para las anotaciones

Una vez clasificadas las muestras, asignaremos una paleta de colores a cada categoría biológica.

# Recomendaciones: 
- https://r-charts.com/es/paletas-colores/#discretas
- https://medium.com/data-science-collective/finding-the-best-color-blind-friendly-palette-on-python-seaborn-0546e4ed33f3

In [ ]:
roi_colors = {
    "IC": "#b30000",
    "HGD": "#2166ac",
    "LGD": "#ffd700",
    "Other": "#aaaaaa"
}

fanconi_colors = {
    "Fanconi": "#762a83",
    "No Fanconi": "#1b7837"
}

tejido_colors = {
    "Stroma": "#ff7f00",
    "Epithelium": "#4d4d4d"
}

## Construcción del clustermap

Una vez definidas las categorías biológicas y sus respectivos colores, podemos proceder a construir los clustermaps a partir de las matrices de distancia.

El procedimiento general consiste en:

1.  Cargar cada matriz de distancia almacenada en formato `.csv`.
2.  Obtener los nombres de las muestras presentes en la matriz.
3.  Asignar automáticamente colores según el tipo de lesión, el estado de Fanconi y el compartimento tisular.
4.  Aplicar clustering jerárquico utilizando el método **average linkage**.
5.  Visualizar simultáneamente la matriz de distancias y el dendrograma resultante.
6.  Guardar la figura para su análisis posterior.

Las barras de colores añadidas a filas y columnas permiten relacionar visualmente los agrupamientos obtenidos con las características biológicas de cada muestra.


In [ ]:
archivos = [
    f for f in os.listdir(ruta)
    if f.endswith(".csv")
]

for archivo in archivos:

    print("Procesando:", archivo)

    df = pd.read_csv(
        os.path.join(ruta, archivo),
        index_col=0
    )

    muestras = df.index.tolist()

    colores = pd.DataFrame({
        "ROI": [roi_colors[roi(x)] for x in muestras],
        "Fanconi": [fanconi_colors[fanconi(x)] for x in muestras],
        "Tejido": [tejido_colors[tejido(x)] for x in muestras]
    }, index=muestras)

    g = sns.clustermap(
        df,
        cmap="_________",
        method="________",
        row_colors=colores,
        col_colors=colores,
        figsize=(12,12),
        yticklabels=False
    )

    plt.setp(
        g.ax_heatmap.get_xticklabels(),
        rotation=90,
        fontsize=6
    )

    g.fig.legend(
        handles=[
            Patch(color="#b30000", label="IC"),
            Patch(color="#2166ac", label="HGD"),
            Patch(color="#ffd700", label="LGD"),
            Patch(color="#762a83", label="Fanconi"),
            Patch(color="#1b7837", label="No Fanconi"),
            Patch(color="#ff7f00", label="Stroma"),
            Patch(color="#4d4d4d", label="Epithelium")
        ],
        bbox_to_anchor=(1.05, 0.5),
        loc="center left"
    )

    nombre = os.path.splitext(archivo)[0]

    plt.savefig(
        os.path.join(
            salida,
            f"clustermap_{nombre}.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

Durante este proceso, el algoritmo reorganiza automáticamente las muestras de acuerdo con sus similitudes topológicas, generando un dendrograma que resume la estructura jerárquica de los agrupamientos. El resultado final es un clustermap anotado que facilita la exploración visual de posibles asociaciones entre la topología de los datos y las características biológicas de las muestras.